# Prepare paired ECG representations

One preparation API replaces duplicated RR and wavelet code. The shared record IDs connect all feature stores to one split.

Install the project first with `pip install -e ".[neural,signal,wavelet,notebooks]"`. Opening this notebook does not start training or download data.


In [ ]:
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
print(f"Project: {ROOT.name}")


## Generate the matched-window cohort

Set `PREPARE = True` only when the raw dataset is present and the target directories are empty. RR-15 establishes the eligible cohort; the other representations must preserve it. Wavelet storage can be large.


In [ ]:
from ecg_afib.features import prepare_dataset
from ecg_afib.splits import create_splits

PREPARE = False
if PREPARE:
    raw = ROOT / "data/raw/training2017"
    processed = ROOT / "data/processed"
    rr15 = prepare_dataset(raw, processed / "rr15", representation="rr", rr_length=15)
    create_splits(rr15, ROOT / "data/splits.json", seed=42)
    prepare_dataset(raw, processed / "rr20", representation="rr", rr_length=20, cohort=processed / "rr15")
    for count in (50, 100):
        prepare_dataset(raw, processed / f"wavelet{count}", representation="wavelet", frequencies=count,
                        cohort=processed / "rr15")
else:
    print("Preparation is disabled. Equivalent CLI commands are in README.md.")


## Verify lineage and representation shapes

The manifest describes a record-aware numeric format. The split validator rejects missing, additional, overlapping, or relabeled records.


In [ ]:
import pandas as pd
from ecg_afib.data import FeatureStore
from ecg_afib.splits import load_splits

summary = []
for name in ("rr15", "rr20", "wavelet50", "wavelet100"):
    folder = ROOT / "data/processed" / name
    if (folder / "manifest.json").exists():
        store = FeatureStore(folder)
        splits = load_splits(store, ROOT / "data/splits.json")
        summary.append({"representation": name, "records": len(store.records), "view_shape": store.view_shape,
                        "train": len(splits["train"]), "validation": len(splits["validation"]), "test": len(splits["test"])})
pd.DataFrame(summary)


## Shared wavelet statistics

The command below uses the same statistics routine for either frequency count, with observed minima and pooled moments:

```bash
ecg-afib stats --dataset data/processed/wavelet50 --output runs/wavelet50_statistics.json
```

For a historical whole-record RR protocol use `--rr-mode chunks` in a separate directory. It is a different observation-window design; see [METHODS.md](../docs/METHODS.md).
